In [4]:
import cv2
import numpy as np
import tensorflow as tf
import serial
import time

# ==============================
# 🔹 Initialize Arduino (optional)
# ==============================
# Change 'COM3' to your Arduino port
try:
    arduino = serial.Serial('COM7', 9600, timeout=1)
    time.sleep(2)
    print("✅ Arduino connected.")
except:
    arduino = None
    print("⚠️ Arduino not connected. Running in simulation mode.")

# ==============================
# 🔹 Load the TFLite Model
# ==============================
interpreter = tf.lite.Interpreter(model_path="eye_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_shape = input_details[0]['shape'][1:3]  # e.g., (24, 24)
print("📏 Model expects input size:", input_shape)

# ==============================
# 🔹 Load Haar Cascades
# ==============================
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

# ==============================
# 🔹 Initialize Variables
# ==============================
cap = cv2.VideoCapture(0)
drowsy_count = 0
threshold = 10  # Number of consecutive frames

# ==============================
# 🔹 Start Detection Loop
# ==============================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        roi_face = frame[y:y+h, x:x+w]
        roi_gray = gray[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray)

        for (ex, ey, ew, eh) in eyes[:2]:  # consider only first two eyes
            eye = roi_face[ey:ey+eh, ex:ex+ew]
            eye = cv2.resize(eye, input_shape)
            eye = eye.astype('float32') / 255.0
            eye = np.expand_dims(eye, axis=0)  # (1, height, width, 3)

            # Run inference
            interpreter.set_tensor(input_details[0]['index'], eye)
            interpreter.invoke()
            pred = interpreter.get_tensor(output_details[0]['index'])[0][0]

            if pred > 0.5:
                label = "Open"
                color = (0, 255, 0)
                drowsy_count = 0
            else:
                label = "Closed"
                color = (0, 0, 255)
                drowsy_count += 1

            cv2.rectangle(roi_face, (ex, ey), (ex+ew, ey+eh), color, 2)
            cv2.putText(frame, label, (x+ex, y+ey-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # Drowsiness alert
    if drowsy_count > threshold:
        cv2.putText(frame, "DROWSINESS ALERT!", (100, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        if arduino:
            arduino.write(b'1')  # send signal to Arduino buzzer
    else: 
        if arduino:
            arduino.write(b'0')

    cv2.imshow("Driver Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
if arduino:
    arduino.close()


✅ Arduino connected.
📏 Model expects input size: [24 24]
